# YSI Datasonde Data Plotter

This notebook plots depth profiles from KOR measurement export files exported from a YSI EXO datasonde. Each detected dive is shown as a separate figure with five panels: depth, dissolved oxygen, temperature (°C and °F), specific conductance, and pH.

---

## How to use

**Step 1** — Export your data from KOR Software as a CSV file.

**Step 2** — Click **Upload CSV(s)** and select one or more KOR export files. You can select multiple files at once — each will be processed separately.

**Step 3** — Click **Plot dives**. Figures will appear automatically for every dive detected in each file.

---

## Adjustable settings

In the configuration cell (just below the upload button) you can change:

- **`DIVE_THRESHOLD_FT`** — depth (ft) above which the sonde is considered at the surface (default: 2 ft)
- **`PAD_MIN`** — minutes of surface data to include before and after each dive (default: 10 min)
- **`MERGE_GAP_MIN`** — dive segments separated by less than this are treated as one dive (default: 15 min)
- **`MIN_MAX_DEPTH_FT`** — ignore excursions shallower than this, e.g. sensor checks at the surface (default: 5 ft)


In [ ]:
import io, re, warnings, pathlib, base64, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

---
## 1  ·  Upload your data file

In [ ]:
# ── Settings you can adjust ──────────────────────────────────────────────────
DIVE_THRESHOLD_FT = 2.0   # depth (ft) above which sonde is 'at surface'
MERGE_GAP_MIN     = 15    # merge dive segments separated by less than this
PAD_MIN           = 10    # minutes of surface data to keep before/after dive
MIN_MAX_DEPTH_FT  = 5.0   # ignore excursions shallower than this
FIG_SIZE          = (14, 13)

# ── Shared axis ranges ───────────────────────────────────────────────────────
# (lo, hi, step) applied to every dive so figures compare directly in a report.
# Set any to None to auto-scale that panel per dive instead. Depth is always
# per-dive: sites differ far too much in depth for a shared axis to be useful.
# A dive whose data falls outside its shared range auto-scales rather than
# being clipped, and the panel is labelled 'independent scale' — SpCond at the
# saline sites (Deep Salt, Swampspedition) is the case that triggers this.
FIXED_DO_MGL = (-0.5, 5.5, 0.5)
FIXED_TEMP_C = (20.0, 28.0, 1.0)
FIXED_SPCOND = (0, 1800, 200)
FIXED_PH     = (6.8, 7.8, 0.1)
COLORS = dict(depth='#2563EB', do='#16A34A', temp='#DC2626', spcond='#D97706', ph='#9333EA')

# ── File upload widget ────────────────────────────────────────────────────────
_upload = widgets.FileUpload(
    accept='.csv',
    multiple=True,
    description='Upload CSV(s)',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='40px'),
)
_status = widgets.Label(
    value='No files uploaded yet.',
    layout=widgets.Layout(align_self='center'),
)
_run_btn = widgets.Button(
    description='Plot dives',
    button_style='success',
    disabled=True,
    layout=widgets.Layout(width='150px', height='40px'),
)
_out = widgets.Output()

# Holds uploaded files as {filename: bytes}
_uploaded_files = {}

def _on_upload(change):
    global _uploaded_files
    val = _upload.value
    if not val:
        return
    # ipywidgets 7: value is a dict keyed by filename
    # ipywidgets 8: value is a tuple of dicts with 'name'/'content' keys
    if isinstance(val, dict):
        _uploaded_files = {
            name: bytes(item['content']) if isinstance(item, dict) else bytes(item)
            for name, item in val.items()
        }
    else:
        _uploaded_files = {item['name']: bytes(item['content']) for item in val}
    n = len(_uploaded_files)
    names = ', '.join(_uploaded_files.keys())
    _status.value = f'✓  {n} file{"s" if n > 1 else ""}: {names}'
    _run_btn.disabled = False

_upload.observe(_on_upload, names='value')

display(widgets.VBox([
    widgets.HBox([_upload, _status]),
    _run_btn,
    _out,
]))
print('Ready ✓  Upload one or more CSVs then click “Plot dives”.')

---
## 2  ·  Load and detect dives

In [ ]:
# Kor exports vary a lot between Kor versions, locales and export configs.
# Column names differ in case, spacing and parenthesisation ('TEMP °C',
# 'Temp °C', 'Temp ( °C )'), so match on a normalised key rather than on the
# literal header text. Dropping non-ASCII also sidesteps the ° / µ characters,
# which decode differently depending on the file's encoding.
def _norm_col(c):
    c = c.replace(' ', ' ').replace('\xa0', ' ')   # narrow / non-breaking spaces
    c = c.replace('(', ' ').replace(')', ' ')
    c = ''.join(ch for ch in c if ch.isascii())
    return re.sub(r'\s+', '', c).upper()


KOR_COL_MAP = {
    'DEPTHFT': 'depth_ft',   'DEPTHM': 'depth_m',
    'TEMPC': 'temp_c',
    'SPCONDS/CM': 'spcond',
    'ODO%SAT': 'do_pct_sat', 'DO%SAT': 'do_pct_sat',
    'ODOMG/L': 'do_mgl',     'DOMG/L': 'do_mgl',
    'ODO%CB': 'do_pct_cb',   'DO%CB': 'do_pct_cb',
    'PH': 'ph',              'PHPH': 'ph',
    'PHMV': 'ph_mv',
    'CABLEPWRV': 'cable_v',  'CABLEPOWERVOLTAGE': 'cable_v',
    'BATTERYV': 'battery_v', 'BATTERYPOWERVOLTAGE': 'battery_v',
}

# Kor writes 12-hour ('4:14:04 PM'), 24-hour ('08:57:54'), 4-digit and 2-digit
# year variants depending on the exporting machine's locale.
KOR_DT_FORMATS = ('%m/%d/%Y %I:%M:%S %p', '%m/%d/%y %I:%M:%S %p',
                  '%m/%d/%Y %H:%M:%S',    '%m/%d/%y %H:%M:%S')


def _kor_header(line):
    """Return (normalised cols, date index, time index) if `line` is a Kor data
    header row, else None. Column order varies between export configs, so find
    DATE/TIME by content rather than assuming DATE comes first."""
    cols = [_norm_col(c) for c in line.split(',')]
    di = ti = None
    for i, c in enumerate(cols):
        if di is None and c.startswith('DATE'):
            di = i
        elif ti is None and c.startswith('TIME') and not c.startswith('TIMEFRACT'):
            ti = i
    return (cols, di, ti) if di is not None and ti is not None else None


def load_kor_csv(file_bytes):
    if not isinstance(file_bytes, (bytes, bytearray)):
        file_bytes = bytes(file_bytes)
    # Kor exports come in several encodings: UTF-16-LE (BOM), UTF-8, or
    # Latin-1/CP1252 (e.g. the ° and µ bytes). Try the BOM/UTF-8 path, then
    # fall back to Latin-1 so files with high bytes still decode.
    enc = 'utf-16-le' if file_bytes[:2] == b'\xff\xfe' else 'utf-8'
    try:
        content = file_bytes.decode(enc)
    except UnicodeDecodeError:
        content = file_bytes.decode('latin-1')
    content = content.lstrip('﻿')
    lines = [ln.rstrip('\r') for ln in content.split('\n')]

    # One export can hold several record blocks, each preceded by its own
    # summary metadata and header row.
    heads = []
    for i, ln in enumerate(lines):
        h = _kor_header(ln)
        if h:
            heads.append((i, h))
    if not heads:
        raise ValueError(
            'no Kor measurement header row found — this does not look like a '
            'Kor measurement export (calibration exports and dive-computer '
            'logs are not supported)')

    frames = []
    for k, (hi, (cols, di, ti)) in enumerate(heads):
        end = heads[k + 1][0] if k + 1 < len(heads) else len(lines)
        rows = [ln for ln in lines[hi + 1:end]
                if ln.strip() and len(ln.split(',')) > di
                and '/' in ln.split(',')[di]]
        if not rows:
            continue
        blk = pd.read_csv(io.StringIO('\n'.join([lines[hi]] + rows)))
        dt = (blk[blk.columns[di]].astype(str).str.strip() + ' '
              + blk[blk.columns[ti]].astype(str).str.strip())
        # Recent Kor builds separate the time from AM/PM with U+202F.
        dt = (dt.str.replace(' ', ' ', regex=False)
                .str.replace('\xa0', ' ', regex=False)
                .str.replace(r'\s+', ' ', regex=True))
        for fmt in KOR_DT_FORMATS:
            try:
                blk['datetime'] = pd.to_datetime(dt, format=fmt)
                break
            except ValueError:
                continue
        else:
            blk['datetime'] = pd.to_datetime(dt, format='mixed')
        blk.columns = [c if c == 'datetime' else _norm_col(c) for c in blk.columns]
        blk = blk.loc[:, ~blk.columns.duplicated()]
        frames.append(blk)

    if not frames:
        raise ValueError('Kor header found but the file contains no data rows')

    df = pd.concat(frames, ignore_index=True).rename(columns=KOR_COL_MAP)
    df = df.sort_values('datetime').set_index('datetime').sort_index()
    df = df.drop(columns=[c for c in df.columns
                          if c.upper().startswith(('DATE', 'TIME'))],
                 errors='ignore')
    # Metric exports record depth in metres; convert to feet so the rest of the
    # notebook (DIVE_THRESHOLD_FT, plots, stats) stays in feet.
    if 'depth_ft' not in df.columns and 'depth_m' in df.columns:
        df['depth_ft'] = pd.to_numeric(df['depth_m'], errors='coerce') * 3.28084
    if 'depth_ft' not in df.columns:
        raise ValueError('no depth column (DEPTH FT / DEPTH M) in this export')
    for c in ('depth_ft', 'temp_c', 'spcond', 'do_mgl',
              'do_pct_sat', 'do_pct_cb', 'ph'):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


def find_dives(df):
    is_diving = df['depth_ft'] > DIVE_THRESHOLD_FT
    groups = (is_diving != is_diving.shift()).cumsum()
    segs = []
    for _, seg in df[is_diving].groupby(groups[is_diving]):
        if seg['depth_ft'].max() >= MIN_MAX_DEPTH_FT:
            segs.append([seg.index[0], seg.index[-1]])
    if not segs:
        return []
    merge_gap = pd.Timedelta(minutes=MERGE_GAP_MIN)
    merged = [segs[0][:]]
    for start, end in segs[1:]:
        if start - merged[-1][1] < merge_gap:
            merged[-1][1] = max(end, merged[-1][1])
        else:
            merged.append([start, end])
    pad = pd.Timedelta(minutes=PAD_MIN)
    return [df.loc[s - pad : e + pad] for s, e in merged
            if len(df.loc[s-pad:e+pad]) > 0]


def _png_link(png, fname, label):
    b64 = base64.b64encode(png).decode()
    return HTML(
        f'<a href="data:image/png;base64,{b64}" download="{fname}" '
        f'style="display:inline-block;padding:4px 14px;background:#4b5563;'
        f'color:white;border-radius:4px;text-decoration:none;font-size:13px;'
        f'margin:2px 0 14px">⬇ {label}</a>'
    )


# ── Wire the 'Plot dives' button ──────────────────────────────────────────────
# all_dives is a list of (filename, dive_index, dataframe) tuples
all_dives = []
_all_pngs = []  # (png_filename, bytes) for every rendered dive

def _on_plot(_):
    global all_dives, _all_pngs
    all_dives = []
    _all_pngs = []
    with _out:
        clear_output(wait=True)
        if not _uploaded_files:
            print('Please upload at least one CSV file first.')
            return
        _sep = '═' * 60
        for fname in sorted(_uploaded_files.keys()):
            print(f'\n{_sep}')
            print(f'  {fname}')
            print(f'{_sep}')
            try:
                df = load_kor_csv(_uploaded_files[fname])
                print(f'  {len(df):,} rows  |  {df.index.min()} → {df.index.max()}')
                dives = find_dives(df)
                if not dives:
                    print('  No dives found — check DIVE_THRESHOLD_FT / MIN_MAX_DEPTH_FT')
                    continue
                print(f'  {len(dives)} dive(s) detected:')
                for i, ds in enumerate(dives, 1):
                    dr = ds[ds['depth_ft'] > DIVE_THRESHOLD_FT]
                    dur = dr.index[-1] - dr.index[0]
                    print(f'    Dive {i}: {dr.index[0].strftime("%Y-%m-%d %H:%M")}  '
                          f'max {ds["depth_ft"].max():.1f} ft  {str(dur).split(".")[0]}')
                    all_dives.append((fname, i, ds))
                print()
                for i, ds in enumerate(dives, 1):
                    try:
                        png = plot_dive(ds, idx=i, source_file=fname)
                        if png:
                            dr = ds[ds['depth_ft'] > DIVE_THRESHOLD_FT]
                            stem = fname.rsplit('.', 1)[0].replace(' ', '_')
                            png_name = (f'{stem}_dive{i:02d}_'
                                        f'{dr.index[0].strftime("%Y%m%d_%H%M")}.png')
                            display(_png_link(png, png_name, f'Save {png_name}'))
                            _all_pngs.append((png_name, png))
                    except Exception as e:
                        import traceback
                        print(f'  ⚠ Error plotting dive {i}: {e}')
                        traceback.print_exc()
            except Exception as e:
                print(f'  ⚠ Error processing {fname}: {e}')

        if len(_all_pngs) > 1:
            zip_buf = io.BytesIO()
            with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
                for png_name, data in _all_pngs:
                    zf.writestr(png_name, data)
            zip_buf.seek(0)
            b64z = base64.b64encode(zip_buf.read()).decode()
            display(HTML(
                f'<a href="data:application/zip;base64,{b64z}" download="dives.zip" '
                f'style="display:inline-block;padding:8px 22px;background:#16A34A;'
                f'color:white;border-radius:6px;text-decoration:none;font-size:15px;'
                f'font-weight:bold;margin-top:12px">⬇ Download all {len(_all_pngs)} dives as ZIP</a>'
            ))

_run_btn.on_click(_on_plot)
print('Functions defined ✓')

---
## 3  ·  Plot all dives

In [ ]:
import matplotlib.ticker as mticker

def _floor(x, step): return np.floor(x / step) * step
def _ceil(x, step):  return np.ceil(x  / step) * step
def _nice_step(span, target):
    """Smallest 1/2/5·10ⁿ step that keeps an axis under `target` ticks."""
    if span <= 0:
        return 1.0
    mag = 10.0 ** np.floor(np.log10(span / target))
    for m in (1, 2, 5):
        if span / (m * mag) <= target:
            return m * mag
    return 10 * mag


def depth_limits(s):
    """0 ft to max depth, 10 ft steps. Past ~150 ft a fixed 10 ft step packs
    the axis with unreadable overlapping labels, so widen it there."""
    hi = _ceil(s.max(), 10)
    step = 10.0 if hi <= 150 else _nice_step(hi, 14)
    return _ceil(s.max(), step), 0.0, step
def do_limits(s):
    lo, hi = _floor(s.min(), 0.1), _ceil(s.max(), 0.1)
    span = hi - lo
    step = 0.1 if span <= 0.5 else 0.2 if span <= 2.0 else 0.5 if span <= 5.0 else 1.0
    return lo, hi, step
def temp_limits(s):
    lo, hi = _floor(s.min(), 1.0), _ceil(s.max(), 1.0)
    return lo, hi, 1.0 if (hi - lo) <= 8 else 2.0
def ph_limits(s):
    lo, hi = _floor(s.min(), 0.1), _ceil(s.max(), 0.1)
    return lo, hi, 0.1 if (hi - lo) <= 1.0 else 0.2
def spcond_limits(s):
    """Floor/ceil to nearest 50 µS/cm; step 50, 100 or 200 depending on range.
    Saline and brackish sites reach tens of thousands of µS/cm, far past the
    point where the fixed ladder yields a readable axis, so compute a step."""
    lo, hi = _floor(s.min(), 50), _ceil(s.max(), 50)
    span = hi - lo
    if span > 1600:
        step = _nice_step(span, 12)
        return _floor(s.min(), step), _ceil(s.max(), step), step
    step = 50 if span <= 200 else 100 if span <= 500 else 200
    return lo, hi, step

def shared_or_auto(fixed, s, auto_fn):
    """Return ((lo, hi, step), independent). Use the shared range when one is
    configured and the dive's data fits inside it; otherwise auto-scale, so
    out-of-range data is never silently clipped off the panel."""
    s = s.dropna()
    if fixed is None or s.empty:
        return auto_fn(s), False
    lo, hi, _ = fixed
    if s.min() < lo or s.max() > hi:
        return auto_fn(s), True
    return fixed, False


def note_independent(ax):
    ax.text(0.995, 0.97, 'independent scale', transform=ax.transAxes,
            fontsize=7.5, va='top', ha='right', style='italic', color='#92400E',
            bbox=dict(boxstyle='round,pad=0.25', fc='#FEF3C7', alpha=0.9, ec='none'))


def apply_limits(ax, lo, hi, step, invert=False):
    ax.set_ylim(hi, lo) if invert else ax.set_ylim(lo, hi)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(step))


def format_xaxis(ax, t0, t1):
    span = t1 - t0
    hours = span.total_seconds() / 3600
    if hours < 2:
        ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0,60,5)))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    elif hours < 6:
        ax.xaxis.set_major_locator(mdates.MinuteLocator(byminute=range(0,60,15)))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    else:
        ax.xaxis.set_major_locator(mdates.HourLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))


def add_dive_time_axis(ax_top, dive_start_ts, xlo_num, xhi_num):
    """Dive-time axis on top of depth panel. Uses numeric xlim directly
    so t=0 aligns exactly with the shading edge and dashed line."""
    ax2 = ax_top.twiny()
    ax2.set_xlim(xlo_num, xhi_num)
    t0_num = mdates.date2num(dive_start_ts)
    def date2min(x): return (x - t0_num) * 24 * 60
    def min2date(m): return t0_num + m / (24 * 60)
    total_min = date2min(xhi_num)
    for step in [2, 5, 10, 15, 20, 30, 60]:
        if total_min / step <= 18:
            tick_step = step; break
    else:
        tick_step = 60
    first_tick = int(np.ceil(date2min(xlo_num) / tick_step)) * tick_step
    tick_mins  = np.arange(max(first_tick, 0), total_min + tick_step, tick_step)
    ax2.set_xticks([min2date(m) for m in tick_mins])
    ax2.set_xticklabels([f'{int(m)}' for m in tick_mins], fontsize=8.5)
    ax2.set_xlabel('Dive time (min)', fontsize=9, labelpad=5)
    ax2.spines[['top','right','left','bottom']].set_visible(False)


def stats_box(ax, s, unit):
    s = s.dropna()
    if s.empty: return
    txt = f'min {s.min():.2f}  ·  mean {s.mean():.2f}  ·  max {s.max():.2f}  {unit}'
    ax.text(0.01, 0.97, txt, transform=ax.transAxes, fontsize=8.5,
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.75, ec='none'))


def plot_dive(ds, idx, save=False, source_file=None):
    diving     = ds[ds['depth_ft'] > DIVE_THRESHOLD_FT]
    dive_start = diving.index[0]
    dive_end   = diving.index[-1]
    duration   = dive_end - dive_start
    title_date = dive_start.strftime('%A, %B %-d %Y  ·  %H:%M')

    # Numeric x-limits — set explicitly on every axis, no auto-margins
    xlo = mdates.date2num(ds.index[0])
    xhi = mdates.date2num(ds.index[-1])
    t0  = mdates.date2num(dive_start)
    t1  = mdates.date2num(dive_end)

    fig = plt.figure(figsize=FIG_SIZE, facecolor='white')
    fig.suptitle(
        f'Dive {idx}  ·  {title_date}\n'
        f'Max depth {ds["depth_ft"].max():.1f} ft  ·  '
        f'Dive duration {str(duration).split(".")[0]}',
        fontsize=13, fontweight='bold', y=1.01)

    gs   = GridSpec(5, 1, figure=fig, hspace=0.10)
    axes = [fig.add_subplot(gs[i]) for i in range(5)]

    for ax in axes:
        ax.set_xlim(xlo, xhi)
        ax.axvspan(xlo, t0, color='#e5e7eb', alpha=0.5, zorder=0, linewidth=0)
        ax.axvspan(t1, xhi, color='#e5e7eb', alpha=0.5, zorder=0, linewidth=0)
        ax.axvline(t0, color='gray', lw=0.8, ls='--', alpha=0.5, zorder=2)
        ax.axvline(t1, color='gray', lw=0.8, ls='--', alpha=0.5, zorder=2)

    ax = axes[0]
    ax.plot(ds.index, ds['depth_ft'], color=COLORS['depth'], lw=1.0, zorder=3)
    ax.fill_between(ds.index, ds['depth_ft'], alpha=0.12, color=COLORS['depth'], zorder=2)
    ax.set_ylabel('Depth (ft)', color=COLORS['depth'], fontsize=10)
    hi, lo, step = depth_limits(diving['depth_ft'])
    apply_limits(ax, lo, hi, step, invert=True)
    ax.tick_params(axis='y', labelcolor=COLORS['depth'])
    stats_box(ax, diving['depth_ft'], 'ft')

    ax = axes[1]
    ax.plot(ds.index, ds['do_mgl'], color=COLORS['do'], lw=1.0, zorder=3)
    ax.set_ylabel('DO (mg/L)', color=COLORS['do'], fontsize=10)
    # Scale to steady-state values: skip first 5 min of descent through surface water
    _do_trim = diving.loc[dive_start + pd.Timedelta(minutes=5):]['do_mgl'].dropna()
    _do_scale = _do_trim if not _do_trim.empty else diving['do_mgl']
    (lo, hi, step), _indep = shared_or_auto(FIXED_DO_MGL, _do_scale, do_limits)
    apply_limits(ax, lo, hi, step)
    if _indep: note_independent(ax)
    ax.tick_params(axis='y', labelcolor=COLORS['do'])
    stats_box(ax, diving['do_mgl'], 'mg/L')

    ax = axes[2]
    ax.plot(ds.index, ds['temp_c'], color=COLORS['temp'], lw=1.0, zorder=3)
    ax.set_ylabel('Temp (°C)', color=COLORS['temp'], fontsize=10)
    (lo, hi, step), _indep = shared_or_auto(FIXED_TEMP_C, diving['temp_c'], temp_limits)
    apply_limits(ax, lo, hi, step)
    if _indep: note_independent(ax)
    ax.tick_params(axis='y', labelcolor=COLORS['temp'])
    stats_box(ax, diving['temp_c'], '°C')
    # Fahrenheit twin axis — limits derived from °C limits so gridlines align
    ax_f = ax.twinx()
    f_lo, f_hi = lo * 9/5 + 32, hi * 9/5 + 32
    ax_f.set_ylim(f_lo, f_hi)
    # MaxNLocator always produces at least 2 clean ticks regardless of range
    ax_f.yaxis.set_major_locator(
        mticker.MaxNLocator(nbins=3, min_n_ticks=2, steps=[1, 2, 5, 10]))
    ax_f.set_ylabel('Temp (°F)', color=COLORS['temp'], fontsize=10)
    ax_f.tick_params(axis='y', labelcolor=COLORS['temp'])
    ax_f.spines[['top','left','bottom']].set_visible(False)
    ax_f.spines['right'].set_visible(False)

    ax = axes[3]
    ax.plot(ds.index, ds['spcond'], color=COLORS['spcond'], lw=1.0, zorder=3)
    ax.set_ylabel('SpCond (µS/cm)', color=COLORS['spcond'], fontsize=10)
    (lo, hi, step), _indep = shared_or_auto(FIXED_SPCOND, diving['spcond'], spcond_limits)
    apply_limits(ax, lo, hi, step)
    if _indep: note_independent(ax)
    ax.tick_params(axis='y', labelcolor=COLORS['spcond'])
    stats_box(ax, diving['spcond'], 'µS/cm')

    ax = axes[4]
    ax.plot(ds.index, ds['ph'], color=COLORS['ph'], lw=1.0, zorder=3)
    ax.axhline(7.0, color='gray', lw=0.6, ls=':', alpha=0.5, zorder=1)
    ax.set_ylabel('pH', color=COLORS['ph'], fontsize=10)
    (lo, hi, step), _indep = shared_or_auto(FIXED_PH, diving['ph'], ph_limits)
    apply_limits(ax, lo, hi, step)
    if _indep: note_independent(ax)
    ax.tick_params(axis='y', labelcolor=COLORS['ph'])
    stats_box(ax, diving['ph'], '')

    for ax in axes[:-1]:
        ax.set_xticklabels([])
        ax.xaxis.set_tick_params(length=0)
    format_xaxis(axes[-1], ds.index[0], ds.index[-1])
    axes[-1].set_xlabel('Time of day', fontsize=10)
    axes[-1].set_xlim(xlo, xhi)

    add_dive_time_axis(axes[0], dive_start, xlo, xhi)
    axes[0].set_xlim(xlo, xhi)

    for ax in axes:
        ax.grid(True, axis='both', ls=':', lw=0.5, color='gray', alpha=0.4, zorder=1)
        ax.spines[['top','right']].set_visible(False)

    fig.align_ylabels(axes)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    _png = buf.read()
    display(fig)
    plt.close(fig)
    return _png


---
## 4  ·  Interactive selector

In [ ]:
if not all_dives:
    print('No dives loaded yet \u2014 upload files and click "Plot dives" above.')
else:
    def make_label(fname, i, ds):
        dr = ds[ds['depth_ft'] > DIVE_THRESHOLD_FT]
        return (f'{fname}  \u00b7  Dive {i}  \u00b7  '
                f'{dr.index[0].strftime("%Y-%m-%d %H:%M")}  '
                f'max {ds["depth_ft"].max():.1f} ft')

    dd = widgets.Dropdown(
        options=[(make_label(f, i, ds), idx)
                 for idx, (f, i, ds) in enumerate(all_dives)],
        description='Dive:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%'))
    btn2 = widgets.Button(description='Plot', button_style='primary',
                          layout=widgets.Layout(width='100px'))
    out2 = widgets.Output()

    def on_click2(_):
        with out2:
            clear_output(wait=True)
            fname, i, ds = all_dives[dd.value]
            png = plot_dive(ds, idx=i, source_file=fname)
            if png:
                dr = ds[ds['depth_ft'] > DIVE_THRESHOLD_FT]
                stem = fname.rsplit('.', 1)[0].replace(' ', '_')
                png_name = (f'{stem}_dive{i:02d}_'
                            f'{dr.index[0].strftime("%Y%m%d_%H%M")}.png')
                display(_png_link(png, png_name, f'Save {png_name}'))

    btn2.on_click(on_click2)
    display(widgets.VBox([dd, btn2, out2]))

---
## 5  ·  Summary statistics

In [ ]:
if not all_dives:
    print('No dives loaded yet — upload files and click “Plot dives” above.')
else:
    PLOT_COLS = [('depth_ft','Depth (ft)'),('do_mgl','DO (mg/L)'),
                 ('do_pct_sat','DO (% sat)'),('temp_c','Temp (°C)'),('ph','pH')]
    for fname, i, ds in all_dives:
        dr = ds[ds['depth_ft'] > DIVE_THRESHOLD_FT]
        dur = dr.index[-1] - dr.index[0]
        print(f'\n{"─"*62}')
        print(f'  {fname}  —  Dive {i}  —  {dr.index[0].strftime("%Y-%m-%d %H:%M")}'
              f'  (duration {str(dur).split(".")[0]}, max {ds["depth_ft"].max():.1f} ft)')
        print(f'{"─"*62}')
        rows = []
        for col, label in PLOT_COLS:
            if col in ds.columns:
                s = ds[col].dropna()
                rows.append({'Parameter':label,'Min':round(s.min(),3),
                             'Mean':round(s.mean(),3),'Max':round(s.max(),3),
                             'Std':round(s.std(),3),'N':len(s)})
        display(pd.DataFrame(rows).set_index('Parameter'))
